## 🎯 Learning Objectives
* Understand the core principles of zero-shot image classification using CLIP.
* Implement a zero-shot image classifier using the Hugging Face Transformers library.
* Apply CLIP to classify images into arbitrary categories not seen during training.
* Interpret the similarity scores and predicted labels from a CLIP model.


## DL03-L12: Exercise - Build a Zero-Shot Image Classifier with CLIP

### Task Description

In this exercise, you will build a zero-shot image classifier using OpenAI's Contrastive Language-Image Pre-training (CLIP) model. CLIP is a powerful multimodal model capable of understanding both images and text, allowing for flexible classification without requiring explicit training on the target categories.

Your goal is to take a set of input images and classify them into a predefined set of candidate text labels. The 'zero-shot' aspect means that the model has never been explicitly trained on these specific image-label pairings, but rather leverages its broad understanding of visual and linguistic concepts.

### Requirements

1.  **Utilize the `transformers` library**: Load a pre-trained CLIP model and its corresponding processor from Hugging Face.
2.  **Prepare input**: You will be provided with a small set of sample images and a list of candidate text labels.
3.  **Implement zero-shot classification logic**: Your code should:
    *   Preprocess the images and tokenize the text labels using the CLIP processor.
    *   Pass the processed inputs through the CLIP model to obtain image and text embeddings.
    *   Compute the similarity (e.g., cosine similarity) between each image embedding and each text embedding.
    *   Convert these similarity scores into probabilities (e.g., using softmax).
    *   Predict the most likely label for each image.
4.  **Display results**: For each image, display the image itself, the predicted label, and the confidence score.

### Evaluation Criteria

*   **Correctness**: The implementation accurately performs zero-shot classification using CLIP.
*   **Clarity**: The code is well-structured, readable, and includes comments where necessary.
*   **Efficiency**: The solution uses appropriate PyTorch and `transformers` functionalities for efficient inference.
*   **Output**: The results clearly show the image, its predicted label, and the associated probability.


In [ ]:
# Ensure you have the necessary libraries installed:
# pip install torch transformers datasets Pillow

import torch
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import requests
from datasets import load_dataset
import matplotlib.pyplot as plt
import numpy as np

# --- Device Configuration ---
# Use CUDA if available, otherwise CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# --- Load Pre-trained CLIP Model and Processor ---
# We'll use a common CLIP model, e.g., 'openai/clip-vit-base-patch32'
# This model is robust and widely used for zero-shot tasks.
model_name = "openai/clip-vit-base-patch32"
processor = CLIPProcessor.from_pretrained(model_name)
model = CLIPModel.from_pretrained(model_name).to(device)
print(f"Loaded CLIP model: {model_name}")

# --- Define Candidate Text Labels ---
# These are the categories we want to classify our images into.
# CLIP's strength is its ability to classify into arbitrary text descriptions.
candidate_labels = [
    "a photo of a cat",
    "a photo of a dog",
    "a photo of an airplane",
    "a photo of a car",
    "a photo of a bird",
    "a photo of a deer",
    "a photo of a frog",
    "a photo of a horse",
    "a photo of a ship",
    "a photo of a truck",
    "a photo of a flower",
    "a photo of a building"
]

# --- Load Sample Images ---
# For this exercise, we'll load a few images from the CIFAR-10 dataset
# using the `datasets` library. This provides diverse, small images.
print("Loading sample images from CIFAR-10 dataset...")
cifar10_dataset = load_dataset("cifar10", split="test")

# Select a few diverse images for demonstration
# We'll pick specific indices to get a mix of classes
image_indices = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90]
# Original CIFAR-10 labels for reference: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck
sample_images = [cifar10_dataset[i]["img"] for i in image_indices]

print(f"Loaded {len(sample_images)} sample images.")

# --- Helper Function to Display Images ---
def display_image_with_prediction(image, predicted_label, confidence):
    """Displays an image along with its predicted label and confidence."""
    plt.figure(figsize=(4, 4))
    plt.imshow(image)
    plt.title(f"Predicted: {predicted_label}\nConfidence: {confidence:.2f}")
    plt.axis("off")
    plt.show()

print("Setup complete. Proceed to implement the zero-shot classifier.")


### Your Implementation: Zero-Shot Image Classifier

Now it's your turn to implement the zero-shot image classification logic. Use the `processor`, `model`, `sample_images`, and `candidate_labels` provided in the setup cell.

Your code should:
1.  Iterate through each `image` in `sample_images`.
2.  For each image, prepare the inputs for the CLIP model (both image and text).
3.  Obtain image and text features (embeddings) from the CLIP model.
4.  Compute the cosine similarity between the image feature and all text features.
5.  Apply softmax to convert similarity scores into probabilities.
6.  Determine the predicted label (the one with the highest probability).
7.  Use the `display_image_with_prediction` helper function to show the results for each image.


In [ ]:
import torch
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import requests
from datasets import load_dataset
import matplotlib.pyplot as plt
import numpy as np

# --- Re-run Setup (for standalone execution or if previous cell was skipped) ---
# In a real notebook, you'd typically run the setup cell once.
# We include it here for robustness if this cell is run independently.

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "openai/clip-vit-base-patch32"
processor = CLIPProcessor.from_pretrained(model_name)
model = CLIPModel.from_pretrained(model_name).to(device)

candidate_labels = [
    "a photo of a cat",
    "a photo of a dog",
    "a photo of an airplane",
    "a photo of a car",
    "a photo of a bird",
    "a photo of a deer",
    "a photo of a frog",
    "a photo of a horse",
    "a photo of a ship",
    "a photo of a truck",
    "a photo of a flower",
    "a photo of a building"
]

cifar10_dataset = load_dataset("cifar10", split="test")
image_indices = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90]
sample_images = [cifar10_dataset[i]["img"] for i in image_indices]

def display_image_with_prediction(image, predicted_label, confidence):
    plt.figure(figsize=(4, 4))
    plt.imshow(image)
    plt.title(f"Predicted: {predicted_label}\nConfidence: {confidence:.2f}")
    plt.axis("off")
    plt.show()

# --- Solution Implementation ---

# 1. Preprocess all candidate text labels once
# Tokenize the text inputs. `return_tensors="pt"` ensures PyTorch tensors.
text_inputs = processor(text=candidate_labels, return_tensors="pt", padding=True).to(device)

# 2. Get text embeddings from the CLIP model
# We use `torch.no_grad()` to disable gradient calculations, saving memory and speeding up inference.
with torch.no_grad():
    text_features = model.get_text_features(**text_inputs)
    # Normalize text features for cosine similarity calculation
    text_features /= text_features.norm(dim=-1, keepdim=True)

print("Starting zero-shot classification...")

# 3. Iterate through each sample image for classification
for i, image in enumerate(sample_images):
    print(f"\nClassifying image {i+1}/{len(sample_images)}...")

    # Preprocess the current image
    # `return_tensors="pt"` ensures PyTorch tensors.
    image_inputs = processor(images=image, return_tensors="pt").to(device)

    # Get image embeddings from the CLIP model
    with torch.no_grad():
        image_features = model.get_image_features(**image_inputs)
        # Normalize image features
        image_features /= image_features.norm(dim=-1, keepdim=True)

    # Compute cosine similarity between image and text features
    # The similarity is a dot product of normalized features.
    similarity = (image_features @ text_features.T).squeeze(0)

    # Convert similarity scores to probabilities using softmax
    # Temperature scaling (model.logit_scale.exp()) is crucial for CLIP's calibrated probabilities.
    logits_per_image = model.logit_scale.exp() * similarity
    probs = logits_per_image.softmax(dim=-1)

    # Get the top prediction
    top_prob, top_idx = probs.topk(1)
    predicted_label_full = candidate_labels[top_idx.item()]
    predicted_confidence = top_prob.item()

    # Extract a cleaner label for display (e.g., "cat" from "a photo of a cat")
    # This is a simple heuristic; more robust parsing might be needed for complex labels.
    predicted_label_display = predicted_label_full.replace("a photo of a ", "").replace("a photo of an ", "")

    # Display the result
    display_image_with_prediction(image, predicted_label_display, predicted_confidence)

print("Zero-shot classification complete for all sample images.")
